In [5]:
from google import genai
from google.genai import types
import time
import os
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv()

# Configuración del cliente
client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

# Nombre del archivo local
FILE_PATH = "manual_tecnico.pdf"

# 1. Verificar si el archivo existe localmente antes de empezar
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"No se encontró el archivo: {FILE_PATH}")

# 2. Buscar si ya existe un store con ese nombre para no duplicar
existing_stores = client.file_search_stores.list()
file_search_store = next(
    (s for s in existing_stores if s.display_name == "mi_biblioteca_avanzada"),
    None
)

if not file_search_store:
    print("Creando nuevo almacén...")
    file_search_store = client.file_search_stores.create(
        config={"display_name": "mi_biblioteca_avanzada"}
    )
else:
    print(f"Usando almacén existente: {file_search_store.name}")

# 3. Subida e indexación
print("Subiendo e indexando archivo...")
operation = client.file_search_stores.upload_to_file_search_store(
    file=FILE_PATH,
    file_search_store_name=file_search_store.name,
    config={
        "display_name": "Manual de Usuario V1",
        "chunking_config": {
            "white_space_config": {
                "max_tokens_per_chunk": 500,
                "max_overlap_tokens": 50
            }
        }
    }
)

while not operation.done:
    print("Procesando...")
    time.sleep(5)
    operation = client.operations.get(operation)

print("¡Todo listo! Realizando consulta...")

# 4. Consulta
response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Resume qué dice el manual sobre sistemas electrónicos o reinicios",
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=[file_search_store.name]
                )
            )
        ]
    )
)

print("\n" + "=" * 50)
print(f"RESPUESTA IA: {response.text}")
print("=" * 50 + "\n")

# 5. Mostrar fuentes (Citas)
if response.candidates[0].grounding_metadata:
    print("FUENTES CONSULTADAS:")
    for chunk in response.candidates[0].grounding_metadata.grounding_chunks:
        if chunk.retrieved_context:
            print(f"- {chunk.retrieved_context.text[:150]}...")

# 6. OPCIONAL: Borrar para limpiar (solo si quieres que el store sea temporal)
client.file_search_stores.delete(
    name=file_search_store.name,
    config={"force": True}
)


Creando nuevo almacén...
Subiendo e indexando archivo...
Procesando...
¡Todo listo! Realizando consulta...

RESPUESTA IA: El manual proporcionado detalla varios sistemas de un vehículo, incluyendo el sistema eléctrico. Se menciona que el sistema eléctrico gestiona la generación, almacenamiento y distribución de energía. Los componentes clave de este sistema son:

*   **Batería:** Almacena energía eléctrica para el arranque del motor y para alimentar sistemas cuando el motor está apagado.
*   **Alternador:** Genera corriente eléctrica mientras el motor está en marcha para recargar la batería.
*   **Motor de arranque:** Un motor eléctrico que inicia el motor de combustión al girar la llave de contacto.
*   **Fusibles y relés:** Protegen los circuitos eléctricos contra sobrecargas y cortocircuitos.
*   **ECU (Unidad de Control):** Actúa como un ordenador que controla y optimiza el funcionamiento del motor y otros sistemas del vehículo.

Aunque el manual describe extensamente los component